# 05 · Comparación de modelos y elección del modelo activo (Fase III)

1. Compara en **validación** NB y SVM (jerárquico y plano) y DistilBERT, si está disponible.
2. Elige el modelo con la regla de la sección 6.6: mayor **F1-macro jerárquico**; si otro queda
   a menos de 2 puntos del mejor y es más rápido, gana el más rápido.
3. Calibra en validación los umbrales de «requiere revisión».
4. Evalúa el modelo elegido en **test, una sola vez**. El resultado se guarda y las
   ejecuciones siguientes lo leen en lugar de recalcularlo (salvo `REEVALUAR_TEST = True`).
5. Publica `models/activo.json`, el modelo que carga la API.

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from clasificador_ml.artefactos import (
    cargar_metadatos,
    cargar_modelo,
    cargar_predicciones,
    cargar_probabilidades,
    carpeta_version,
    publicar_activo,
)
from clasificador_ml.evaluacion import (
    evaluar,
    guardar_json,
    matrices_area_por_dominio,
    matriz_confusion,
    reporte_por_clase,
)
from clasificador_ml.rutas import DATOS_PROCESADOS, MODELOS
from clasificador_ml.seleccion import (
    curva_revision,
    elegir_umbrales,
    predicciones_desde_probabilidades,
    seleccionar_modelo,
)

DIRECTORIO_MODELOS = Path(os.environ.get('SICDA_MODELOS', MODELOS))
VERSIONES = {'complement_nb': None, 'linear_svc': None, 'distilbert': None}  # None = la más reciente
ACCURACY_MINIMA_ACEPTADOS = 0.90  # accuracy jerárquica exigida a lo que no va a revisión
REEVALUAR_TEST = False
PUBLICAR = True

categorias = json.loads((DATOS_PROCESADOS / 'categorias.json').read_text(encoding='utf-8'))
nombre_dominio = {d['id']: d['codigo'] for d in categorias['dominios']}
nombre_area = {a['id']: a['nombre'] for d in categorias['dominios'] for a in d['areas']}
area_a_dominio = {a['id']: d['id'] for d in categorias['dominios'] for a in d['areas']}
areas_por_dominio = {d['id']: [a['id'] for a in d['areas']] for d in categorias['dominios']}

## Candidatos en validación

In [ ]:
def con_reales(prediccion, particion):
    reales = particion.set_index('id_documento').loc[prediccion['id_documento']]
    prediccion = prediccion.copy()
    prediccion.insert(1, 'dominio_real', reales['id_dominio'].to_numpy())
    prediccion.insert(2, 'area_real', reales['id_area'].to_numpy())
    return prediccion


val = pd.read_parquet(DATOS_PROCESADOS / 'val.parquet')
candidatos = {}
for algoritmo in ('complement_nb', 'linear_svc'):
    m = cargar_metadatos(algoritmo, VERSIONES[algoritmo], DIRECTORIO_MODELOS)
    for tipo, info in m['tipos'].items():
        candidatos[f'{algoritmo} · {tipo}'] = {
            'algoritmo': algoritmo, 'version': m['version'], 'tipo': tipo,
            'predicciones': cargar_predicciones(algoritmo, tipo, m['version'], DIRECTORIO_MODELOS),
            'tiempo_ms': info['tiempo_prediccion']['media_ms'],
        }
try:
    m = cargar_metadatos('distilbert', VERSIONES['distilbert'], DIRECTORIO_MODELOS)
    probabilidades = cargar_probabilidades('distilbert', 'val', m['version'], DIRECTORIO_MODELOS)
    candidatos['distilbert · plano'] = {
        'algoritmo': 'distilbert', 'version': m['version'], 'tipo': 'plano',
        'predicciones': con_reales(predicciones_desde_probabilidades(probabilidades, area_a_dominio), val),
        'tiempo_ms': m['tiempo_prediccion']['media_ms'],
    }
except FileNotFoundError:
    print('DistilBERT no está en models/distilbert/: se comparan solo NB y SVM.')

filas = {}
for nombre, c in candidatos.items():
    p = c['predicciones']
    met = evaluar(p['dominio_real'], p['dominio_pred'], p['area_real'], p['area_pred'])
    filas[nombre] = {
        'version': c['version'],
        'f1_jerarquico': met['jerarquico']['f1_macro'],
        'accuracy_jerarquica': met['jerarquico']['accuracy'],
        'f1_dominio': met['dominio']['f1_macro'],
        'f1_area': met['area']['f1_macro'],
        'f1_ponderado_area': met['area']['f1_ponderado'],
        'tiempo_ms': c['tiempo_ms'],
    }
comparacion = pd.DataFrame(filas).T.sort_values('f1_jerarquico', ascending=False)
comparacion = comparacion.astype({c: float for c in comparacion.columns if c != 'version'})
comparacion.round(4)

In [ ]:
ELEGIDO = seleccionar_modelo(comparacion, metrica='f1_jerarquico', tiempo='tiempo_ms')
elegido = candidatos[ELEGIDO]
mejor = comparacion['f1_jerarquico'].idxmax()
print(f'Mejor F1-macro jerárquico: {mejor} ({comparacion.loc[mejor, "f1_jerarquico"]:.4f})')
print(f'Modelo elegido: {ELEGIDO} ({comparacion.loc[ELEGIDO, "f1_jerarquico"]:.4f}, '
      f'{comparacion.loc[ELEGIDO, "tiempo_ms"]:.0f} ms por resumen)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for nombre, fila in comparacion.iterrows():
    ax.scatter(fila['tiempo_ms'], fila['f1_jerarquico'], s=60, c='C3' if nombre == ELEGIDO else 'C0')
    ax.annotate(nombre, (fila['tiempo_ms'], fila['f1_jerarquico']), textcoords='offset points',
                xytext=(5, 5), fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('ms por resumen (CPU, escala log)')
ax.set_ylabel('F1-macro jerárquico (validación)')
ax.set_title('Calidad frente a velocidad')
plt.tight_layout()

## Calibración de los umbrales de revisión (validación)

Se recorren `umbral_dominio` ∈ [0,30; 0,90] y `umbral_conjunta` ∈ [0,10; 0,80] y se elige la
combinación que envía **menos** documentos a revisión con una accuracy jerárquica de al menos
`ACCURACY_MINIMA_ACEPTADOS` en los que se aceptan automáticamente.

In [ ]:
curva = curva_revision(elegido['predicciones'])
umbral = elegir_umbrales(curva, ACCURACY_MINIMA_ACEPTADOS)
UMBRALES = {'umbral_dominio': float(umbral['umbral_dominio']),
            'umbral_conjunta': float(umbral['umbral_conjunta'])}
diseno = curva[(curva['umbral_dominio'] == 0.60) & (curva['umbral_conjunta'] == 0.40)]
pd.concat([diseno.assign(caso='diseño (0,60 / 0,40)'),
           umbral.to_frame().T.assign(caso='calibrado')]).set_index('caso').round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
puntos = ax.scatter(100 * curva['porcentaje_revision'], curva['accuracy_aceptados'],
                    c=curva['umbral_conjunta'], cmap='viridis', s=12)
ax.scatter(100 * diseno['porcentaje_revision'], diseno['accuracy_aceptados'], marker='s', c='k',
           label='diseño (0,60 / 0,40)')
ax.scatter(100 * umbral['porcentaje_revision'], umbral['accuracy_aceptados'], marker='*', s=200,
           c='C3', label='calibrado')
ax.axhline(ACCURACY_MINIMA_ACEPTADOS, ls='--', c='grey', lw=1)
ax.set_xlabel('% de documentos que requieren revisión')
ax.set_ylabel('Accuracy jerárquica de los aceptados')
ax.legend()
fig.colorbar(puntos, label='umbral conjunta')
ax.set_title(f'Umbrales · {ELEGIDO}')
plt.tight_layout()

## Evaluación en test (una sola vez)

In [ ]:
algoritmo, version, tipo = elegido['algoritmo'], elegido['version'], elegido['tipo']
carpeta = carpeta_version(algoritmo, version, DIRECTORIO_MODELOS)
ruta_informe = carpeta / f'evaluacion_test_{tipo}.json'
ruta_predicciones = carpeta / f'predicciones_test_{tipo}.parquet'

if ruta_informe.is_file() and not REEVALUAR_TEST:
    print(f'El test ya se evaluó: se lee {ruta_informe.name} y no se vuelve a calcular.')
    informe_test = json.loads(ruta_informe.read_text(encoding='utf-8'))
    prediccion_test = pd.read_parquet(ruta_predicciones)
else:
    test = pd.read_parquet(DATOS_PROCESADOS / 'preprocesado' / 'test.parquet')
    if algoritmo == 'distilbert':
        probabilidades = cargar_probabilidades('distilbert', 'test', version, DIRECTORIO_MODELOS)
        prediccion_test = predicciones_desde_probabilidades(probabilidades, area_a_dominio, **UMBRALES)
    else:
        modelo = cargar_modelo(algoritmo, tipo, version, DIRECTORIO_MODELOS)
        modelo.preprocesador = None  # se usa texto_procesado, ya calculado con el mismo preprocesador
        modelo.set_params(**UMBRALES)
        prediccion_test = modelo.predecir_niveles(test['texto_procesado'].tolist())
        prediccion_test.insert(0, 'id_documento', test['id_documento'].to_numpy())
    prediccion_test = con_reales(prediccion_test, test)
    acierto = ((prediccion_test['dominio_pred'] == prediccion_test['dominio_real'])
               & (prediccion_test['area_pred'] == prediccion_test['area_real']))
    aceptados = ~prediccion_test['requiere_revision']
    informe_test = {
        'modelo': ELEGIDO,
        'algoritmo': algoritmo,
        'version': version,
        'tipo': tipo,
        'fecha': datetime.now().isoformat(timespec='seconds'),
        'umbrales': UMBRALES,
        'metricas': evaluar(prediccion_test['dominio_real'], prediccion_test['dominio_pred'],
                            prediccion_test['area_real'], prediccion_test['area_pred']),
        'porcentaje_revision': float(prediccion_test['requiere_revision'].mean()),
        'accuracy_aceptados': float(acierto[aceptados].mean()),
        'tiempo_ms': float(comparacion.loc[ELEGIDO, 'tiempo_ms']),
    }
    guardar_json(informe_test, ruta_informe)
    prediccion_test.to_parquet(ruta_predicciones, index=False)

pd.DataFrame(informe_test['metricas']).round(4)

In [ ]:
metricas_test = informe_test['metricas']
pd.DataFrame({
    'meta': ['F1-macro dominio ≥ 0,85', 'F1-macro área ≥ 0,65', '< 2 s por resumen'],
    'valor': [metricas_test['dominio']['f1_macro'], metricas_test['area']['f1_macro'],
              informe_test['tiempo_ms'] / 1000],
    'cumple': [metricas_test['dominio']['f1_macro'] >= 0.85,
               metricas_test['area']['f1_macro'] >= 0.65,
               informe_test['tiempo_ms'] < 2000],
}).set_index('meta').round(4)

In [ ]:
print(f"Requieren revisión en test: {100 * informe_test['porcentaje_revision']:.1f} % · "
      f"accuracy jerárquica de los aceptados: {informe_test['accuracy_aceptados']:.4f}")
matriz = matriz_confusion(prediccion_test['dominio_real'], prediccion_test['dominio_pred'],
                          sorted(nombre_dominio), nombre_dominio)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(matriz.div(matriz.sum(axis=1), axis=0), annot=True, fmt='.2f', cmap='Blues', ax=ax)
ax.set_title(f'Dominio en test · {ELEGIDO}')
plt.tight_layout()

### Áreas por dominio (test)

Accuracy de área dentro de cada dominio real; la columna `otro dominio` cuenta los documentos
que se enviaron a un dominio equivocado.

In [ ]:
matrices = matrices_area_por_dominio(
    prediccion_test['dominio_real'], prediccion_test['dominio_pred'],
    prediccion_test['area_real'], prediccion_test['area_pred'], areas_por_dominio, nombre_area,
)
resumen = pd.DataFrame({
    nombre_dominio[d]: {
        'areas': len(areas_por_dominio[d]),
        'documentos': int(m.to_numpy().sum()),
        'accuracy_area': m.to_numpy()[:, :-1].trace() / m.to_numpy().sum(),
        'a_otro_dominio': m['otro dominio'].sum() / m.to_numpy().sum(),
    }
    for d, m in matrices.items()
}).T
resumen.round(3)

In [ ]:
DOMINIO_A_MOSTRAR = 0  # id del dominio (0 = CS)
m = matrices[DOMINIO_A_MOSTRAR]
fig, ax = plt.subplots(figsize=(0.45 * m.shape[1] + 3, 0.45 * m.shape[0] + 2))
sns.heatmap(m.div(m.sum(axis=1), axis=0), cmap='Blues', annot=m.shape[0] <= 20, fmt='.2f', ax=ax)
ax.set_title(f'Áreas de {nombre_dominio[DOMINIO_A_MOSTRAR]} en test (proporción por fila)')
plt.tight_layout()

In [ ]:
reporte = reporte_por_clase(prediccion_test['area_real'], prediccion_test['area_pred'], nombre_area)
reporte.head(15).round(3)

## Publicar el modelo activo

In [ ]:
if not PUBLICAR:
    print('PUBLICAR = False: no se modifica models/activo.json.')
elif algoritmo == 'distilbert':
    print('Ganó DistilBERT. La API carga modelos .joblib: hace falta un envoltorio para el '
          'modelo de transformers (Fase IV) antes de publicarlo.')
else:
    ruta = publicar_activo(algoritmo, version, tipo, UMBRALES, metricas_test, DIRECTORIO_MODELOS,
                           tiempo_ms=informe_test['tiempo_ms'])
    print(json.dumps(json.loads(ruta.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))

La decisión (modelo, umbrales y métricas de test) se registra en
`docs/decisiones/0003-modelo-activo.md`.